In [2]:
from config import Config
from DataManager import DataManager
from DataPreprocessor import Preprocessor
from FeatureSelection import FeatureSelector
from ModelManager import ModelManager
from ModelEvaluation import Evaluator

from DataExplorer import Explorer #new


from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
cfg = Config()
dm = DataManager(cfg)
preprocessor = Preprocessor(cfg)

gt_df = dm.load_ground_truth()
wl = dm.load_wavelengths()

X_train, valid_train= preprocessor.process_data(dm, image_ids=dm.get_data_indices(cfg.TRAIN_DATA_PATH), file_path=cfg.TRAIN_DATA_PATH)
X_test, _= preprocessor.process_data(dm, image_ids=dm.get_data_indices(cfg.TEST_DATA_PATH), file_path=cfg.TEST_DATA_PATH)

y_train = gt_df.loc[valid_train,:].values


processed 100/1732 images...
processed 200/1732 images...
processed 300/1732 images...
processed 400/1732 images...
processed 500/1732 images...
processed 600/1732 images...
processed 700/1732 images...
processed 800/1732 images...
processed 900/1732 images...
processed 1000/1732 images...
processed 1100/1732 images...
processed 1200/1732 images...
processed 1300/1732 images...
processed 1400/1732 images...


KeyboardInterrupt: 

In [31]:
X_train[:100]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
image_data = dm.load_image(1700)#1700 image - takie samo jak wcześniej
Explorer.compare_masked_vs_not_masked(image_data, 50 )


In [ ]:
Explorer.plot_single_image_spectrum(image_data, wl)

In [ ]:
#Calculating and plotting first, second and third order derivatives of avg spectral reflectance curve
Explorer.plot_image_derivatives(image_data, wl)

In [ ]:
#Calculating correlation between bands of the example image
Explorer.plot_band_correlation_matrix(image_data)

In [ ]:
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=cfg.RANDOM_SEED)

In [32]:
selector = FeatureSelector(cfg, n_features=25) #tu zmieniłem z 15 na 25 bo był jakiś błąd
selector.fit(X_train, y_train)

X_train_rfe = selector.transform(X_train)
X_test_rfe = selector.transform(X_test)

Fitting estimator with 150 features.
Fitting estimator with 145 features.
Fitting estimator with 140 features.
Fitting estimator with 135 features.
Fitting estimator with 130 features.
Fitting estimator with 125 features.
Fitting estimator with 120 features.
Fitting estimator with 115 features.
Fitting estimator with 110 features.
Fitting estimator with 105 features.
Fitting estimator with 100 features.
Fitting estimator with 95 features.
Fitting estimator with 90 features.
Fitting estimator with 85 features.
Fitting estimator with 80 features.
Fitting estimator with 75 features.
Fitting estimator with 70 features.
Fitting estimator with 65 features.
Fitting estimator with 60 features.
Fitting estimator with 55 features.
Fitting estimator with 50 features.
Fitting estimator with 45 features.
Fitting estimator with 40 features.
Fitting estimator with 35 features.
Fitting estimator with 30 features.


In [33]:
models = ModelManager(cfg, model_type='xgb')

models.tune_fit(X_train_rfe, y_train, verbose=True)
y_pred = models.predict(X_test_rfe)


Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [P] parameter: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.6}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [K] parameter: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.6}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [Mg] parameter: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 1000, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.6}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [pH] parameter: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.6}



In [34]:
#generating submission file
dm.save_submission(y_pred, filename='submission_new')


Succesfully saved submission file to: ../Submissions\submission_new.csv


In [ ]:
evaluator = Evaluator(cfg)
y_baseline = ModelManager.get_baseline_predictions(X_train=X_train_rfe, y_train=y_train,X_test=X_test_rfe)
#evaluator.calculate_hyperview_score(y_test, y_pred, y_baseline)

P              0.970280
K              0.945541
Mg             0.914177
pH             0.848254
FINAL_SCORE    0.919563
dtype: float64

In [ ]:
models_new = ModelManager(cfg)
models_new.load_models('best_rf_model.pkl')

y_pred_new = models_new.predict(X_test_rfe)
#evaluator.calculate_hyperview_score(y_test, y_pred_new, y_baseline)

In [6]:
#test wyboru pliku potem do wyrzucenua

from Users import Farmer
cfg = Config()
farmer = Farmer(cfg, user_id=1) #

print("odpalamy okno")

# metoda ktora pozwala w konie windowsa wybrac obraz
farmer.load_data_from_desktop()

if farmer.data is not None:
    print("\nWczytano obraz.")
    print(f"Wymiary danych: {farmer.data.shape}")
else:
    print("\nAnulowano wybór lub wystąpił błąd.")


odpalamy okno

Anulowano wybór lub wystąpił błąd.
